In [1]:
# from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
# import torch
# from PIL import Image


/home/ubuntu/additional_drive/shwan_data/auto_annotations/llavaenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# model_id = "llava-hf/llava-v1.6-mistral-7b-hf"

In [3]:
# processor = LlavaNextProcessor.from_pretrained(model_id)

# model = LlavaNextForConditionalGeneration.from_pretrained(
#     model_id,
#     torch_dtype=torch.float16,
#     device_map="auto",          # puts it on GPU if available
#     low_cpu_mem_usage=True,
# )
# model.eval()


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:12<00:00,  3.05s/it]
/home/ubuntu/additional_drive/shwan_data/auto_annotations/llavaenv/lib/python3.10/site-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['model.image_newline']
  warnings.warn(
Some parameters are on the meta device because they were offloaded to the cpu.


LlavaNextForConditionalGeneration(
  (model): LlavaNextModel(
    (vision_tower): CLIPVisionModel(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(577, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)

In [5]:
from PIL import Image
import json
import torch
import matplotlib.pyplot as plt
import numpy as np

image_path = "../../temp_yolo/all_dataset_singleclass/images/train/19_crime_rose_frame_1.png"   # ← change this
image = Image.open(image_path).convert("RGB")


In [6]:
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {
                "type": "text",
                "text": (
                    "Annotate the product in the image. "
                    "Return response in ONLY this JSON format:\n\n"
                    "{\n"
                    "  \"category\": \"\",\n"
                    "  \"color\": \"\",\n"
                    "  \"material\": \"\",\n"
                    "  \"tags\": [],\n"
                    "  \"polygon\": [[x1, y1], [x2, y2], ...]\n"
                    "}\n\n"
                    "Polygon must outline the product. "
                    "Use image coordinates with top-left = (0,0)."
                )
            }
        ]
    }
]

prompt = processor.apply_chat_template(
    conversation, add_generation_prompt=True
)

inputs = processor(image, prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.0,
        do_sample=False
    )

response_text = processor.decode(output_ids[0], skip_special_tokens=True)
print(response_text)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST]  
Annotate the product in the image. Return response in ONLY this JSON format:

{
  "category": "",
  "color": "",
  "material": "",
  "tags": [],
  "polygon": [[x1, y1], [x2, y2], ...]
}

Polygon must outline the product. Use image coordinates with top-left = (0,0). [/INST] ```json
{
  "category": "Beverage",
  "color": "Pink",
  "material": "Glass",
  "tags": ["Alcoholic", "Wine"],
  "polygon": [[0.208,0.000,0.800,0.988]]
}
``` 


In [9]:
print(response_text)

[INST]  
Annotate the product in the image. Return response in ONLY this JSON format:

{
  "category": "",
  "color": "",
  "material": "",
  "tags": [],
  "polygon": [[x1, y1], [x2, y2], ...]
}

Polygon must outline the product. Use image coordinates with top-left = (0,0). [/INST] ```json
{
  "category": "Beverage",
  "color": "Pink",
  "material": "Glass",
  "tags": ["Alcoholic", "Wine"],
  "polygon": [[0.208,0.000,0.800,0.988]]
}
``` 


In [7]:
# Try to extract JSON safely
try:
    if "{" in response_text:
        json_str = response_text[response_text.index("{"):]
        annotation = json.loads(json_str)
    else:
        raise Exception("No JSON found")

except Exception as e:
    print("JSON parsing failed:", e)
    print("Raw response:", response_text)
    annotation = None

annotation


JSON parsing failed: Expecting value: line 6 column 16 (char 82)
Raw response: [INST]  
Annotate the product in the image. Return response in ONLY this JSON format:

{
  "category": "",
  "color": "",
  "material": "",
  "tags": [],
  "polygon": [[x1, y1], [x2, y2], ...]
}

Polygon must outline the product. Use image coordinates with top-left = (0,0). [/INST] ```json
{
  "category": "Beverage",
  "color": "Pink",
  "material": "Glass",
  "tags": ["Alcoholic", "Wine"],
  "polygon": [[0.208,0.000,0.800,0.988]]
}
``` 
